# From workflow to agent

Notebook 7 closed the biggest hole — the assistant can now *do* things — but three
of the six gaps are still open, and one is only half-closed:

| # | Gap | State after nb 7 |
| --- | --- | --- |
| 1 | No action tools | **fixed** (two of them; this notebook adds a third and an ordering rule between them) |
| 2 | System prompt is a switch statement | **open** — the prompt still says "if decision X, say Y" |
| 3 | `extract_return_request` conflates extraction and dispatch | **open** — `execute_tool` still routes it into the rules engine |
| 4 | Unbounded loop, no budget | **half, and staying that way** — `MAX_ITERATIONS` exists; the token budget needs a bigger model, see the next cell |
| 5 | Validation failure executes anyway | **fixed** |
| 6 | Only one `stop_reason` handled | **half** — `max_tokens` handled; `refusal`, `pause_turn`,
  `model_context_window_exceeded` are not, and `lookup_order` still reports
  "no order found" as a *successful* result |

This notebook closes #2, #3, and #6. The central change is #2 and #3 together,
because they're the same change seen from two sides:

**`process_return_request` is deleted.** In its place is `check_return_eligibility`
— a *fact source*. It answers "is this order returnable, and why not" and returns
nothing about what to say or which tool to call next. The system prompt carries
the policy as context instead of a branch table, and Claude picks the tool.

That sounds like giving up control. It isn't: the gates move from the prompt into
the **validator**, and they now check *facts* ("has eligibility actually been
checked for this order, and did it come back returnable?") rather than a decision
string the rules engine handed down. A prompt is a suggestion; a validator that
refuses to run the tool is enforcement. That's the difference between a model
following a script and a model operating inside a boundary.

## Model: staying on `claude-haiku-4-5`

Same model as notebook 7, on purpose. The change here is architectural — where
the decisions live — and swapping the model at the same time would make it
impossible to tell which change caused what.

That does leave half of gap #4 open, and it's worth knowing why rather than
forgetting about it. `MAX_ITERATIONS` is a *backstop*: it cuts the model off
mid-thought when it trips. A **task budget** is the other half — the server tells
Claude how many tokens it has for the whole turn and counts down, so it paces
itself and wraps up gracefully instead of being guillotined. The two fail in
different directions, which is why you eventually want both.

Task budgets need a bigger model (Opus 5, Fable 5, Sonnet 5, Opus 4.7/4.8) — on
Haiku the parameter isn't available at all. So that half waits for a later
notebook, and the loop below is capped only by iterations.

In [17]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Tools: facts, extraction, and levers — kept apart

Four categories now, and the separation is the point:

| Tool | Kind | Changes the world? |
| --- | --- | --- |
| `lookup_order` | fact | no |
| `check_return_eligibility` | fact | no |
| `extract_return_request` | extraction | no |
| `create_return_authorization` | action | writes an RMA |
| `issue_refund` | action | writes a refund |
| `escalate_to_human` | action | writes a ticket |

`extract_return_request` now does **only** what its name says: it records the
request as structured data and the result echoes that data back. It no longer
doubles as a hidden call into a rules engine, so the model's tool choice and the
system's control flow can't silently diverge the way they did in notebook 6.

`check_return_eligibility` is the replacement for `process_return_request`. Note
what its result does *not* contain: no `decision`, no instruction, no next step.
Just facts — does the order exist, what's its status, how old is it, is it inside
the window, and a list of blocking reasons. What to do about those facts is
Claude's call.

`issue_refund` is new, and it exists to make one thing concrete: **actions can
have ordering constraints between them.** A refund without an RMA is a refund for
goods nobody asked the customer to send back. The validator enforces the order;
the prompt only mentions it.

In [18]:
tools = [
    {
        "name": "lookup_order",
        "description": "Look up an order by its order ID and return its item, status, order date, and total. Call this whenever the customer references an order number or asks about the state of an existing order. Returns an error result if no such order exists.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID, e.g. A1001",
                }
            },
            "required": ["order_id"],
        },
    },
    {
        # FACT SOURCE. Replaces notebook 7's process_return_request. It reports
        # what is true about the order against the return policy; it does not
        # say what to do. Deciding is Claude's job, and the action tools'
        # validators are what hold the line.
        "name": "check_return_eligibility",
        "description": "Check an order against the return policy and report the facts: whether the order exists, its status, how many days old it is, whether it falls inside the return window, and any reasons a return is blocked. This tells you what is TRUE, not what to do — decide that yourself and explain it to the customer. Call this before authorizing any return.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID to check, e.g. A1001.",
                }
            },
            "required": ["order_id"],
            "additionalProperties": False,
        },
    },
    {
        # EXTRACTION ONLY. The result is the extraction echoed back plus a note
        # of what's still missing. It routes nowhere and decides nothing.
        "name": "extract_return_request",
        "description": "Record a customer's return or refund request as structured data. This is bookkeeping, not a decision: the result is your own extraction echoed back, so you can see what you captured and what is still missing. It does not authorize anything. Use it once you understand what the customer wants.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The order ID mentioned by the customer, e.g. A1001, or null if they haven't given one.",
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "defective_item",
                        "wrong_item_shipped",
                        "changed_mind",
                        "billing_dispute",
                        "late_delivery",
                        "unclear",
                        "other",
                    ],
                    "description": "The customer's stated reason. Use 'unclear' if not stated clearly enough to classify.",
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "How urgent the request sounds, based on tone and content.",
                },
                "missing_information": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": [
                            "order_id",
                            "reason",
                            "item_condition",
                            "preferred_resolution",
                        ],
                    },
                    "description": "Which pieces of information the customer has not yet provided.",
                },
                "summary": {
                    "type": "string",
                    "description": "One-sentence, neutral summary of what the customer wants.",
                },
            },
            "required": [
                "order_id",
                "reason",
                "urgency",
                "missing_information",
                "summary",
            ],
            "additionalProperties": False,
        },
    },
    {
        # ACTION. Writes an RMA record to returns.json.
        "name": "create_return_authorization",
        "description": "Authorize a return and issue an RMA number. This writes a real record. Only call it once check_return_eligibility has confirmed the order is returnable — the call is rejected otherwise. Do not tell the customer their return is approved until this returns ok: true.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID being returned, e.g. A1001.",
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "defective_item",
                        "wrong_item_shipped",
                        "changed_mind",
                        "late_delivery",
                        "other",
                    ],
                    "description": "The approved return reason.",
                },
                "note": {
                    "type": "string",
                    "description": "One-sentence note for the warehouse team about the condition or context of the return.",
                },
            },
            "required": ["order_id", "reason", "note"],
            "additionalProperties": False,
        },
    },
    {
        # ACTION. Writes a refund record. Ordering constraint: an RMA must exist
        # for this order first — enforced by the validator, not by the prompt.
        "name": "issue_refund",
        "description": "Refund the customer for an authorized return. Requires an RMA for the same order to exist already — call create_return_authorization first. The amount must match the order total exactly. This writes a real record; don't tell the customer they've been refunded until it returns ok: true.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order being refunded, e.g. A1001.",
                },
                "amount": {
                    "type": "number",
                    "description": "The refund amount. Must equal the order total.",
                },
            },
            "required": ["order_id", "amount"],
            "additionalProperties": False,
        },
    },
    {
        # ACTION. Files a ticket and hands the case to a human.
        "name": "escalate_to_human",
        "description": "File a ticket for a human specialist and hand the case off. Use your judgement: escalate when the request needs a human — billing disputes, angry customers, policy exceptions you can't grant, or anything the other tools can't resolve. Don't tell the customer you're escalating until this returns ok: true.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The order ID if known, or null if the customer never gave a valid one.",
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "billing_dispute",
                        "high_urgency",
                        "order_not_found",
                        "policy_exception",
                        "other",
                    ],
                    "description": "Why this needs a human.",
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "How urgently a human should pick this up.",
                },
                "summary": {
                    "type": "string",
                    "description": "Neutral summary for the specialist who picks this up, including anything the customer told you that isn't in the order record (e.g. an email address).",
                },
            },
            "required": ["order_id", "reason", "urgency", "summary"],
            "additionalProperties": False,
        },
    },
]

## Case state

Two things changed from notebook 7's `case`.

**`last_decision` is gone.** It was the rules engine telling the action tools what
they were allowed to do, and it had a hole: the gate for `escalate_to_human`
accepted `tool_input["reason"] == "order_not_found"` as an alternative to a real
decision, which meant the model could escalate any case at any time just by
naming that reason. A gate the caller can satisfy by asserting something isn't a
gate.

In its place, `facts` — a per-order record of what `check_return_eligibility`
*actually returned*. The validator reads that. The model can't fabricate an entry
in it; only running the fact tool creates one.

**`state` is no longer binary.** A return is authorized, *then* refunded: two
separate side effects with an order between them. So `return_authorized` is a
real state that isn't terminal, and the loop keeps going until it reaches one
that is.

In [19]:
import json
import uuid
from datetime import date
from pathlib import Path

ORDERS_FILE = Path("orders.json")
RETURNS_FILE = Path("returns.json")
REFUNDS_FILE = Path("refunds.json")
ESCALATIONS_FILE = Path("escalations.json")

# The task is over when the case reaches one of these. `return_authorized` is
# deliberately NOT here — an authorized return still owes the customer a refund,
# so the loop should keep working.
TERMINAL_STATES = {"refunded", "escalated"}

# Facts about the policy, not judgements. These feed check_return_eligibility.
NON_RETURNABLE_STATUSES = {"cancelled", "processing"}
RETURN_WINDOW_DAYS = 30


def new_case() -> dict:
    """Fresh task state for one customer conversation.

    `facts` maps an order_id to the eligibility result that check_return_eligibility
    actually returned for it. The action validators read this — so a return can
    only be authorized against eligibility that was really checked, not against a
    claim in the tool call.
    """
    return {"state": "open", "actions": [], "facts": {}}


case = new_case()


def find_order(order_id: str) -> dict | None:
    """Look up one order in orders.json, case-insensitively."""
    orders = json.loads(ORDERS_FILE.read_text())
    return next((o for o in orders if o["order_id"].lower() == order_id.lower()), None)


def append_record(path: Path, record: dict) -> None:
    """Append a record to a JSON array file, creating it if it doesn't exist."""
    records = json.loads(path.read_text()) if path.exists() else []
    records.append(record)
    path.write_text(json.dumps(records, indent=2) + "\n")


def action_for(tool_name: str, order_id: str | None = None) -> dict | None:
    """Find an action already taken on this case, optionally for one order."""
    return next(
        (
            a
            for a in case["actions"]
            if a["tool"] == tool_name
            and (order_id is None or a["order_id"] == order_id)
        ),
        None,
    )

## Implementations

`check_return_eligibility` is worth reading closely against notebook 7's
`process_return_request`. Same policy, different contract: it returns
`returnable` plus a list of `blocking_reasons`, and nothing else. It has no
opinion about escalation, about whether to ask the customer a question, or about
what to say. Those were the parts that made the old function a director.

It also **records what it found** into `case["facts"]`. That record — not a
decision string, not the model's assertion — is what the action validators check.

Each action tool still re-checks its own preconditions before writing. The prompt
says it, the validator enforces it, and the tool verifies it: three layers,
because each one is the only thing standing there if the layer above is bypassed.

One more fix here, from gap #6: `lookup_order` and the action tools now return
`(payload, is_error)`. "No order found" is a *failure*, and the loop turns that
into a `tool_result` with `is_error: True` so the model treats it as one rather
than as a fact it just learned.

In [20]:
def lookup_order(order_id: str) -> tuple[str, bool]:
    """Look up an order. Returns (result_json, is_error)."""
    order = find_order(order_id)
    if order is None:
        # Gap #6: this is a failed call, not a successfully-retrieved fact.
        return f"No order found with ID '{order_id}'.", True

    return json.dumps(order), False


def check_return_eligibility(order_id: str) -> tuple[str, bool]:
    """FACT SOURCE: report what is true about this order under the return policy.

    Deliberately returns no decision, no next step, and no suggested wording.
    Claude reads the facts and chooses; the action validators hold the boundary.
    """
    order = find_order(order_id)
    if order is None:
        return json.dumps({"order_id": order_id, "exists": False}), True

    days_since_order = (date.today() - date.fromisoformat(order["order_date"])).days

    blocking_reasons = []
    if order["status"] in NON_RETURNABLE_STATUSES:
        blocking_reasons.append(
            f"order status is '{order['status']}' — it was never delivered, "
            "so there is nothing to send back"
        )
    if days_since_order > RETURN_WINDOW_DAYS:
        blocking_reasons.append(
            f"ordered {days_since_order} days ago, outside the "
            f"{RETURN_WINDOW_DAYS}-day return window"
        )

    facts = {
        "order_id": order["order_id"],
        "exists": True,
        "status": order["status"],
        "item": order["item"],
        "total": order["total"],
        "days_since_order": days_since_order,
        "return_window_days": RETURN_WINDOW_DAYS,
        "returnable": not blocking_reasons,
        "blocking_reasons": blocking_reasons,
    }

    # This is the record the action validators trust. Only running this tool
    # creates it — the model can't assert its way past the gate.
    case["facts"][order["order_id"].upper()] = facts

    return json.dumps(facts), False


def extract_return_request(**extraction) -> tuple[str, bool]:
    """EXTRACTION ONLY: echo the structured request back.

    Notebook 6 and 7 routed this straight into the business rules, so the model
    thought it was recording data while the system treated the recording as a
    command. Here it records, and that's all.
    """
    return json.dumps({"recorded": extraction}), False


def create_return_authorization(order_id: str, reason: str, note: str) -> tuple[str, bool]:
    """ACTION: write an RMA record to returns.json."""
    facts = case["facts"].get(order_id.upper())
    # Last line of defence. The validator already checked this; if it were ever
    # bypassed, this is what stops a bad RMA reaching disk.
    if not facts or not facts["returnable"]:
        return json.dumps({
            "ok": False,
            "error": f"Order '{order_id}' is not confirmed returnable — no RMA created.",
        }), True

    existing = action_for("create_return_authorization", facts["order_id"])
    if existing:
        # Idempotency: a retrying model must not open two RMAs.
        return json.dumps({
            "ok": True,
            "rma_id": existing["id"],
            "note": "This order already has an authorized return; returning the existing RMA.",
        }), False

    rma_id = f"RMA-{uuid.uuid4().hex[:8].upper()}"
    append_record(RETURNS_FILE, {
        "rma_id": rma_id,
        "order_id": facts["order_id"],
        "item": facts["item"],
        "refund_amount": facts["total"],
        "reason": reason,
        "note": note,
        "created_at": date.today().isoformat(),
    })

    case["state"] = "return_authorized"
    case["actions"].append({
        "tool": "create_return_authorization",
        "order_id": facts["order_id"],
        "id": rma_id,
    })

    return json.dumps({
        "ok": True,
        "rma_id": rma_id,
        "item": facts["item"],
        "refund_amount": facts["total"],
        "ship_back_within_days": 14,
    }), False


def issue_refund(order_id: str, amount: float) -> tuple[str, bool]:
    """ACTION: write a refund record and close the case."""
    rma = action_for("create_return_authorization", order_id.upper())
    if rma is None:
        return json.dumps({
            "ok": False,
            "error": f"No authorized return exists for '{order_id}' — refund refused.",
        }), True

    existing = action_for("issue_refund", order_id.upper())
    if existing:
        return json.dumps({
            "ok": True,
            "refund_id": existing["id"],
            "note": "This order was already refunded; returning the existing refund.",
        }), False

    refund_id = f"REF-{uuid.uuid4().hex[:8].upper()}"
    append_record(REFUNDS_FILE, {
        "refund_id": refund_id,
        "rma_id": rma["id"],
        "order_id": order_id.upper(),
        "amount": amount,
        "created_at": date.today().isoformat(),
    })

    case["state"] = "refunded"
    case["actions"].append({
        "tool": "issue_refund",
        "order_id": order_id.upper(),
        "id": refund_id,
    })

    return json.dumps({
        "ok": True,
        "refund_id": refund_id,
        "amount": amount,
        "settles_within_days": 5,
    }), False


def escalate_to_human(order_id, reason: str, urgency: str, summary: str) -> tuple[str, bool]:
    """ACTION: file a specialist ticket and close the case."""
    existing = action_for("escalate_to_human")
    if existing:
        return json.dumps({
            "ok": True,
            "ticket_id": existing["id"],
            "note": "This case is already escalated; returning the existing ticket.",
        }), False

    ticket_id = f"ESC-{uuid.uuid4().hex[:6].upper()}"
    append_record(ESCALATIONS_FILE, {
        "ticket_id": ticket_id,
        "order_id": order_id,
        "reason": reason,
        "urgency": urgency,
        "summary": summary,
        "created_at": date.today().isoformat(),
    })

    case["state"] = "escalated"
    case["actions"].append({
        "tool": "escalate_to_human",
        "order_id": order_id.upper() if order_id else None,
        "id": ticket_id,
    })

    return json.dumps({
        "ok": True,
        "ticket_id": ticket_id,
        "response_within_hours": 4 if urgency == "high" else 24,
    }), False


ACTION_TOOLS = {"create_return_authorization", "issue_refund", "escalate_to_human"}

_IMPLEMENTATIONS = {
    "lookup_order": lambda i: lookup_order(i["order_id"]),
    "check_return_eligibility": lambda i: check_return_eligibility(i["order_id"]),
    "extract_return_request": lambda i: extract_return_request(**i),
    "create_return_authorization": lambda i: create_return_authorization(**i),
    "issue_refund": lambda i: issue_refund(**i),
    "escalate_to_human": lambda i: escalate_to_human(**i),
}


def execute_tool(tool_name: str, tool_input: dict) -> tuple[str, bool]:
    """Dispatch a tool_use block to its implementation. Returns (result, is_error).

    Note there is no routing cleverness left here: each name maps to the function
    of the same name. Notebook 6's bug — extract_return_request secretly invoking
    the business rules — was possible because this function had opinions.
    """
    impl = _IMPLEMENTATIONS.get(tool_name)
    if impl is None:
        return f"Unknown tool '{tool_name}'.", True

    return impl(tool_input)

## Validation: where the control actually lives

With the switch statement gone from the prompt, this is the file that keeps the
assistant honest. Same two layers as before — schema, then semantics — but the
semantic checks on the action tools now ask about **facts on the case**, not
about a decision the rules engine issued:

- `create_return_authorization` requires `case["facts"][order_id]["returnable"]`
  to be `True` — meaning `check_return_eligibility` was actually run for *this*
  order and actually came back clean.
- `issue_refund` requires an RMA for the same order, and an amount matching the
  order total to the cent. That's the ordering constraint between the two
  actions, enforced rather than requested.
- `escalate_to_human` is deliberately **not** gated on a decision. Escalation is
  the judgement call we're handing to the model; gating it on a rules verdict is
  what made notebook 7 a workflow. What's still enforced is that it's honest work
  — a real order ID if one is given, and a summary a human can actually act on.

That's the shape of the inversion: the model chooses freely among the levers, and
the levers themselves refuse to move when the facts don't support it.

In [21]:
import re
from jsonschema import Draft7Validator

_SCHEMA_VALIDATORS = {
    tool["name"]: Draft7Validator(tool["input_schema"]) for tool in tools
}

_ORDER_ID_RE = re.compile(r"^[A-Za-z]\d+$")


def validate_schema(tool_name: str, tool_input: dict) -> list[str]:
    """Validate tool_input against the tool's own JSON schema."""
    validator = _SCHEMA_VALIDATORS.get(tool_name)
    if validator is None:
        return [f"Unknown tool '{tool_name}'."]

    return [error.message for error in validator.iter_errors(tool_input)]


def validate_return_request_semantics(tool_input: dict) -> list[str]:
    """Consistency checks the schema can't express for extract_return_request."""
    errors = []

    order_id = tool_input.get("order_id")
    missing = tool_input.get("missing_information", [])
    summary = tool_input.get("summary", "")

    if order_id is not None:
        if not _ORDER_ID_RE.match(order_id.strip()):
            errors.append(
                f"order_id '{order_id}' doesn't look like a valid order ID "
                "(expected a letter followed by digits, e.g. A1001)."
            )
        if "order_id" in missing:
            errors.append(
                "order_id is set but 'order_id' is also listed in missing_information — "
                "these are inconsistent."
            )
    elif "order_id" not in missing:
        errors.append(
            "order_id is null but 'order_id' is not listed in missing_information."
        )

    if not summary.strip():
        errors.append("summary is empty — provide a one-sentence neutral summary.")
    elif len(summary.split()) < 3:
        errors.append("summary is too short to be a meaningful one-sentence summary.")

    return errors


def validate_action_semantics(tool_name: str, tool_input: dict) -> list[str]:
    """Gates on the three write tools. These run before anything touches disk."""
    errors = []

    if case["state"] in TERMINAL_STATES:
        errors.append(
            f"This case is already in state '{case['state']}' — it's finished. "
            "Don't take another action on it."
        )

    order_id = tool_input.get("order_id")

    if tool_name == "create_return_authorization":
        # The gate is a FACT, not a decision: eligibility must have actually been
        # checked for this order, and must have actually come back returnable.
        facts = case["facts"].get((order_id or "").upper())
        if facts is None:
            errors.append(
                f"No eligibility check on record for '{order_id}'. Call "
                "check_return_eligibility for this order before authorizing a return."
            )
        elif not facts["returnable"]:
            errors.append(
                f"Order '{facts['order_id']}' is not returnable: "
                + "; ".join(facts["blocking_reasons"])
                + ". Explain this to the customer rather than authorizing a return. "
                "If they say they were charged anyway, that's a billing dispute — "
                "escalate_to_human instead."
            )

    if tool_name == "issue_refund":
        # Ordering constraint between two actions, enforced rather than requested.
        rma = action_for("create_return_authorization", (order_id or "").upper())
        if rma is None:
            errors.append(
                f"No return has been authorized for '{order_id}'. Call "
                "create_return_authorization first — a refund without an RMA means "
                "nobody ever asked the customer to send the item back."
            )

        order = find_order(order_id) if order_id else None
        amount = tool_input.get("amount")
        if order is not None and amount is not None:
            # Money: exact match only, compared in cents to dodge float drift.
            if round(amount * 100) != round(order["total"] * 100):
                errors.append(
                    f"amount {amount} doesn't match the order total {order['total']}. "
                    "Refund the exact order total."
                )

    if tool_name == "escalate_to_human":
        # No decision gate — escalation is the model's judgement call. What's
        # checked is that the ticket is honest and usable.
        if order_id is not None and find_order(order_id) is None:
            errors.append(
                f"order_id '{order_id}' doesn't match any order — pass null "
                "instead and explain the situation in the summary."
            )
        if len(tool_input.get("summary", "").split()) < 5:
            errors.append(
                "summary is too thin — a human is going to read this cold, so "
                "include what the customer wants and anything they told you."
            )

    return errors


def validate_tool_call(tool_name: str, tool_input: dict) -> list[str]:
    """Schema validation, then (if that passes) tool-specific semantic validation."""
    errors = validate_schema(tool_name, tool_input)
    if errors:
        return errors

    if tool_name == "extract_return_request":
        return validate_return_request_semantics(tool_input)

    if tool_name in ACTION_TOOLS:
        return validate_action_semantics(tool_name, tool_input)

    return []

## The system prompt: policy, not a branch table

Here is notebook 7's prompt, in outline:

> - `need_order_id`: ask for their order ID.
> - `order_not_found`: let them know that order ID doesn't match anything.
> - `need_clarification`: ask why they'd like to return the item.
> - `not_returnable`: say so plainly and kindly.
> - `escalate_to_specialist`: call escalate_to_human.
> - `approve_return`: call create_return_authorization.

Every branch of the rules engine, mapped to a line for Claude to say. That's a
workflow with an LLM renderer on top — and it's why the model went off-script the
moment reality didn't match a branch.

What replaces it is the *policy* — the same rules, stated as facts about the
business rather than as a dispatch table — plus what the levers are and what the
hard constraints are. No mapping from state to sentence. The model reads the
situation and picks.

The constraints that remain in the prompt are the ones that are genuinely about
integrity rather than routing: don't claim something happened before the tool
said it did, and don't invent policy. Both are also enforced elsewhere — the
prompt is where they're explained, not where they're enforced.

In [22]:
SYSTEM_PROMPT = """You are a shop assistant for an online store, handling returns and refunds.

## Return policy

- Items can be returned within 30 days of the order date.
- Only orders that were actually delivered or shipped can be returned. A
  cancelled or still-processing order was never delivered, so there is nothing
  to send back.
- An approved return gets an RMA number first, then a refund for the full order
  total. The refund is never issued without an RMA.
- Billing disputes — a customer charged for something they didn't receive, or
  charged twice — are not returns. A human handles those.

## How to work

You have tools that tell you facts (lookup_order, check_return_eligibility) and
tools that change things (create_return_authorization, issue_refund,
escalate_to_human). Establish the facts, then decide what the situation calls
for. Nothing dictates your next step — that's your judgement.

check_return_eligibility reports what is true about an order under the policy
above. It does not tell you what to do. Read the blocking reasons it returns and
explain them to the customer in your own words.

Use escalate_to_human when a human is genuinely needed: billing disputes, an
angry customer whose problem you can't solve, a policy exception you can't
grant, or an order ID that doesn't exist and the customer can't correct. Don't
escalate as a way of avoiding a conversation you can have yourself.

If you're missing something you need — usually the order ID, sometimes the
reason — just ask for it.

## Hard rules

1. Never tell the customer something has happened until the tool has returned
   ok: true. Not "I'm escalating this now", not "your return is approved", not
   "you've been refunded". If a tool fails or is rejected, say plainly that it
   didn't go through — never describe a failed action as if it succeeded.
2. Don't invent policy. If the rules above don't cover the situation, say so and
   escalate rather than making up an answer.
3. One resolution per case: either the return is authorized and refunded, or the
   case is escalated to a human.

Be warm and direct. Keep replies short — this is a support chat, not a letter.
"""

# Appended to the system prompt for the final message when the case closed. The
# model can't see the loop, so without this it signs off with "anything else I
# can help with?" — a question the customer can't answer, because the chat is
# about to exit.
CLOSED_CASE_NOTE = """CLOSING MESSAGE. This case is now closed and this is the last thing you will
say — the conversation ends here and you will not see a reply. Write a short
sign-off that confirms what was done, quotes the RMA, refund, or ticket number,
and says what happens next and roughly when. Do not offer further help, do not
ask whether there's anything else, and do not ask any question at all.
"""

# The other exits: the case is still open, we just couldn't get further this
# turn. Here a question is exactly right — we need something from the customer.
UNRESOLVED_NOTE = """CLOSING MESSAGE. You could not complete this request, and no action was taken —
no return was authorized, no refund was issued, no ticket was filed. Say so
honestly without inventing a reason, and never imply anything happened. Tell the
customer in one or two sentences what you need from them to move forward.
"""

## The loop: handling every stop reason

`MAX_ITERATIONS` carries over from notebook 7 unchanged — it's still the only
thing bounding the loop, for the reason in the model note above.

What's new is **gap #6**. Notebook 7 handled exactly one stop reason
(`max_tokens`) and fell through on the rest, which meant an unexpected one was an
unhandled exception rather than a message to the customer. `_STOP_HANDLERS`
covers the ones that need distinct treatment:

| `stop_reason` | What it means | What we do |
| --- | --- | --- |
| `refusal` | safety classifiers declined; `content` may be empty | stop the turn, say so plainly |
| `max_tokens` | truncated — may hold a half-formed `tool_use` with no matching `tool_result`, which makes the *next* request invalid | drop the turn |
| `model_context_window_exceeded` | the conversation no longer fits | stop; the fix is compaction, not a retry |
| `pause_turn` | a long-running server tool paused | resend to resume (not reachable here — no server tools — but the branch is where it belongs) |
| `tool_use` | normal | run the tools |
| anything else | normal end of turn | return the text |

Note `refusal` is checked **before** reading `response.content`: on a refusal the
content array can be empty, so code that indexes into it blindly raises. Haiku
refuses far less than the frontier models do, so this branch will rarely fire
here — but "rarely" is exactly the kind of path that breaks in production and
never in testing.

In [23]:
MAX_VALIDATION_RETRIES = 2
# Backstop on the tool-use loop: a hard stop, felt by us, not by the model.
# Nothing here is allowed to run forever.
MAX_ITERATIONS = 10
# Hard per-response cap. The model never sees it — that's what makes it a
# guillotine rather than a budget.
MAX_TOKENS = 1024

messages = []


def text_of(response) -> str:
    """Join a response's text blocks into one string. Never raises.

    A response can legitimately have no text block — a refusal, a truncated turn,
    or a turn that was pure tool_use — and the caller still needs something to
    show the customer.
    """
    parts = [block.text for block in response.content if block.type == "text"]
    joined = "\n".join(parts).strip()
    return joined or f"(no text in response; stop_reason={response.stop_reason})"


def call_model(messages: list, *, system: str, tool_choice: dict | None = None):
    """One request. Every call in this notebook goes through here."""
    kwargs = {
        "model": model,
        "max_tokens": MAX_TOKENS,
        "system": system,
        "tools": tools,
        "messages": messages,
    }
    if tool_choice is not None:
        kwargs["tool_choice"] = tool_choice

    return client.messages.create(**kwargs)


def finish(messages: list, reason: str, closing_note: str) -> str:
    """End the turn with a closing message and no ability to act.

    tool_choice "none" means Claude can only write text here, so this exit can't
    start a new action or a new tool-call cycle. The closing note tells it which
    ending it's writing — otherwise the sign-off contradicts what the system did.
    """
    print(f"[turn ending: {reason}]")
    final = call_model(
        messages,
        system=SYSTEM_PROMPT + "\n\n" + closing_note,
        tool_choice={"type": "none"},
    )
    messages.append({"role": "assistant", "content": final.content})
    return text_of(final)

# Each handler returns the text to show the customer and ends the turn. Only
# "tool_use", "pause_turn", and ordinary end-of-turn fall through to the loop.

def _handle_refusal(response, messages) -> str:
    details = getattr(response, "stop_details", None)
    category = getattr(details, "category", None) if details else None
    print(f"[refusal] safety classifiers declined (category={category})")
    # The turn is not appended: a refusal may carry empty content, and there is
    # nothing here worth keeping in the history.
    return (
        "Sorry — I'm not able to help with that one. If this is about an order, "
        "tell me the order ID and what went wrong and I'll take another look."
    )


def _handle_max_tokens(response, messages) -> str:
    # A truncated turn can contain a half-formed tool_use block with no matching
    # tool_result, which makes the NEXT request invalid. Drop it.
    print("[warning] response truncated at max_tokens — dropping that turn")
    return "Sorry — I ran out of room mid-reply. Could you send that again?"


def _handle_context_exceeded(response, messages) -> str:
    print("[warning] context window exceeded — the conversation no longer fits")
    return (
        "Sorry — this conversation has got too long for me to keep track of. "
        "Could you start a fresh one with your order ID?"
    )


_STOP_HANDLERS = {
    "refusal": _handle_refusal,
    "max_tokens": _handle_max_tokens,
    "model_context_window_exceeded": _handle_context_exceeded,
}


def send_message(messages: list) -> str:
    """Call Claude with the current conversation, running the tool-use loop until
    one of the exits below. Always returns text."""
    validation_retries = 0
    pause_resumes = 0

    for _ in range(MAX_ITERATIONS):
        response = call_model(messages, system=SYSTEM_PROMPT)

        # Checked before touching response.content — on a refusal it can be empty.
        handler = _STOP_HANDLERS.get(response.stop_reason)
        if handler is not None:
            return handler(response, messages)

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "pause_turn":
            # A server-side tool paused mid-run. Resending resumes it — but cap
            # the resumes so a stuck pause can't spin.
            pause_resumes += 1
            if pause_resumes > 3:
                return finish(messages, "too many pause_turn resumes", UNRESOLVED_NOTE)
            continue

        if response.stop_reason != "tool_use":
            # Exit 1: Claude stopped on its own.
            return text_of(response)

        tool_results = []
        gave_up = False
        for block in response.content:
            if block.type != "tool_use":
                continue

            print(f"[tool call] {block.name}({block.input})")
            errors = validate_tool_call(block.name, block.input)

            if errors:
                validation_retries += 1
                give_up = validation_retries > MAX_VALIDATION_RETRIES
                gave_up = gave_up or give_up
                label = (
                    "giving up"
                    if give_up
                    else f"retry {validation_retries}/{MAX_VALIDATION_RETRIES}"
                )
                print(f"[validation failed, {label}] {errors}")

                error_text = "Your tool call was rejected:\n" + "\n".join(
                    f"- {error}" for error in errors
                )
                # Either way the tool does NOT run. An exhausted retry budget
                # means "don't", not "do it anyway".
                error_text += (
                    "\nDon't call this tool again — explain the situation to the customer instead."
                    if give_up
                    else "\nFix the input, or choose a different course of action."
                )
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": error_text,
                    "is_error": True,
                })
                continue

            result, is_error = execute_tool(block.name, block.input)
            if block.name in ACTION_TOOLS:
                print(f"[action] {block.name} -> {result}")

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result,
                "is_error": is_error,
            })

        messages.append({"role": "user", "content": tool_results})

        # Exit 2: an action tool reached a terminal state — the task is over.
        if case["state"] in TERMINAL_STATES:
            return finish(messages, f"case closed ({case['state']})", CLOSED_CASE_NOTE)

        # Exit 3: we've stopped executing this tool, so leaving the model free to
        # call it again is how you get an infinite retry cycle.
        if gave_up:
            return finish(messages, "validation retry budget exhausted", UNRESOLVED_NOTE)

    # Exit 4: backstop. Still asking for tools after MAX_ITERATIONS rounds.
    return finish(messages, f"hit MAX_ITERATIONS={MAX_ITERATIONS}", UNRESOLVED_NOTE)

## Chat

Things worth trying, and what to watch for:

- **"I want to return A1005, wrong item"** — A1005 is cancelled. Watch the model
  read `blocking_reasons` and explain it *in its own words*: there's no branch in
  the prompt telling it what to say any more.
- **Then: "but I was charged for it"** — this is the case notebook 7 mishandled
  first time round. There's no `not_returnable → billing_dispute` rule to follow;
  the policy says billing disputes go to a human and the model has to make that
  connection itself.
- **"Return A1007"** — delivered, but ordered outside the 30-day window. A second
  kind of blocking reason, same fact tool, no new branch.
- **"Return A1001, it's defective"** — the happy path, and now a two-step one:
  RMA, then refund. Try telling it to just refund you without the return and
  watch the validator refuse.

Check `returns.json`, `refunds.json`, and `escalations.json` afterwards. Call
`case.update(new_case())` (or restart the kernel) for a fresh case.

In [24]:
while True:
    try:
        user_input = input("You: ")
        print(f"User: {user_input}")
    except (EOFError, KeyboardInterrupt):
        break

    # VS Code's input box returns "" for both Escape and a blank Enter, so blank
    # input doubles as the way to quit here.
    if not user_input.strip() or user_input.lower() in ("quit", "exit"):
        break

    messages.append({"role": "user", "content": user_input})
    # send_message appends the assistant turn(s) itself, including any
    # intermediate tool_use/tool_result turns from the tool loop.
    reply = send_message(messages)

    print(f"Assistant: {reply}")

    if case["state"] in TERMINAL_STATES:
        print(f"\n--- case closed ({case['state']}): {case['actions']} ---")
        break

User: Hello I want my money back for order A1005
[tool call] lookup_order({'order_id': 'A1005'})
[tool call] check_return_eligibility({'order_id': 'A1005'})
Assistant: I can see why you'd want your money back, but unfortunately order A1005 can't be returned for two reasons:

1. **The order was cancelled** — it was never delivered, so there's nothing to send back.
2. **It's outside the return window** — the order is 39 days old, and we allow returns within 30 days.

If you were charged for this order and didn't receive it, that's a billing issue rather than a return — I'd like to escalate this to a specialist who can look into that for you. Can you confirm: were you charged for this order even though it was cancelled?
User: Yes, I was chaged for that
[tool call] escalate_to_human({'order_id': 'A1005', 'reason': 'billing_dispute', 'urgency': 'medium', 'summary': 'Customer was charged $29.99 for order A1005 (Laptop Stand) which was cancelled and never delivered. Requesting refund for the 